<h1 style="text-align:center; color:#0b3954;">
 Analyse des accidents corporels de la route en France<br>
<span style="font-size:0.65em; color:#555;">Base BAAC - Année 2023</span>
</h1>

<p style="text-align:center; font-style:italic; color:#555;">
Projet réalisé dans le cadre du cours <b>Python pour la Data Science</b><br>
Enseignant : Lino Galiana — ENSAE
</p>

---

## 📑 Table des matières

1. [Contexte, problématique et objectifs](#1-contexte-problématique-et-objectifs)
2. [Importation des bibliothèques](#2-importation-des-bibliothèques)
3. [Récupération des données via l'API data.gouv.fr](#3-récupération-des-données-via-lapi-datagouvfr)
4. [Nettoyage et enrichissement](#4-nettoyage-et-enrichissement)
5. [Exploration statistique générale](#5-exploration-statistique-générale)
6. [Analyses univariées](#6-analyses-univariées)
    - 6.1 [Temporelles (mois, jour, heure)](#61-temporelles-mois-jour-heure)
    - 6.2 [Conditions environnementales](#62-conditions-environnementales)
    - 6.3 [Caractéristiques des usagers](#63-caractéristiques-des-usagers)
    - 6.4 [Équipements de sécurité](#64-équipements-de-sécurité)
7. [Analyses bivariées (croisements avec la gravité)](#7-analyses-bivariées-croisements-avec-la-gravité)
8. [Cartographie](#8-cartographie)
    - 8.1 [Heatmap nationale](#81-heatmap-nationale)
    - 8.2 [Choroplèthe par département](#82-choroplèthe-par-département)
    - 8.3 [Accidents mortels (cluster)](#83-accidents-mortels-cluster)
9. [Modélisation — Prédire la gravité d'un accident](#9-modélisation--prédire-la-gravité-dun-accident)
    - 9.1 [Préparation](#91-préparation)
    - 9.2 [Régression logistique (baseline)](#92-régression-logistique-baseline)
    - 9.3 [Random Forest équilibrée](#93-random-forest-équilibrée)
    - 9.4 [Comparaison des modèles](#94-comparaison-des-modèles)
    - 9.5 [Importance des variables](#95-importance-des-variables)
10. [Conclusion](#10-conclusion)

---

## 1. Contexte, problématique et objectifs
<a id="1-contexte-problématique-et-objectifs"></a>

###  Contexte
La sécurité routière est un enjeu majeur de santé publique en France. En 2023, selon l'Observatoire
National Interministériel de la Sécurité Routière (ONISR), plus de **3 000 personnes** ont perdu la
vie sur les routes françaises, et plusieurs dizaines de milliers ont été blessées. Chaque accident
corporel est recensé dans la **base BAAC** (Bulletin d'Analyse des Accidents Corporels), librement
diffusée sur [data.gouv.fr](https://www.data.gouv.fr/fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2023/).

Cette base est structurée en quatre fichiers complémentaires reliés par l'identifiant
`Num_Acc` :

| Fichier      | Contenu                                          |
|--------------|--------------------------------------------------|
| `caract`     | Caractéristiques générales (date, lieu, météo)   |
| `lieux`      | Description de la route (type, trafic, surface)  |
| `usagers`    | Victimes (âge, sexe, gravité, équipement)        |
| `vehicules`  | Véhicules impliqués                              |

###  Problématique
> **Quels sont les principaux facteurs — temporels, environnementaux, humains et routiers —
> associés aux accidents corporels en France en 2023, et peut-on prédire la gravité
> d'un accident à partir de ces variables ?**

###  Objectifs
1. **Collecte & nettoyage** : récupérer les 4 fichiers BAAC 2023 via l'API de data.gouv.fr
   et construire une base analytique cohérente.
2. **Analyse descriptive** : identifier les dimensions temporelles, spatiales et
   humaines expliquant la sinistralité.
3. **Cartographie** : localiser les zones à risque à différentes échelles (heatmap,
   département, accidents mortels).
4. **Modélisation** : construire un classifieur capable de prédire si un accident sera
   **grave** (blessé hospitalisé ou tué) à partir de variables disponibles au moment des faits,
   en comparant une régression logistique et une forêt aléatoire.

## 2. Importation des bibliothèques
<a id="2-importation-des-bibliothèques"></a>

In [ ]:
# Dossier de sortie pour les cartes HTML
import os
os.makedirs('outputs', exist_ok=True)

In [ ]:
# Manipulation & analyse
import pandas as pd
import numpy as np
import requests
from io import BytesIO

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Cartographie
import folium
from folium.plugins import HeatMap, MarkerCluster

# Modélisation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score
)

# Confort
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("Set2")
print("✓ Bibliothèques chargées.")

## 3. Récupération des données via l'API data.gouv.fr
<a id="3-récupération-des-données-via-lapi-datagouvfr"></a>

Plutôt que de télécharger manuellement les fichiers, on interroge l'API publique de
`data.gouv.fr` pour récupérer dynamiquement les URL des quatre fichiers CSV de l'année cible.
Cela garantit que le notebook reste **reproductible** même si les liens changent.

In [ ]:
DATASET_ID = "53698f4ca3a729239d2036df"  # jeu de données BAAC
ANNEE = "2023"

def list_csv_resources(dataset_id: str) -> list:
    """Retourne la liste des ressources CSV du jeu de données."""
    api_url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/"
    resp = requests.get(api_url, timeout=30)
    resp.raise_for_status()
    resources = resp.json()["resources"]
    return [r for r in resources if (r.get("format") or "").upper() == "CSV"]

def find_url(csv_resources, keyword: str, annee: str) -> str:
    """Renvoie l'URL du CSV dont le titre contient `keyword` et `annee`."""
    match = next(
        (r["url"] for r in csv_resources
         if keyword in r["title"].lower() and annee in r["title"]),
        None,
    )
    if match is None:
        raise RuntimeError(f"Fichier '{keyword}' {annee} introuvable.")
    return match

csv_resources = list_csv_resources(DATASET_ID)
file_keys = {"caract": "caract", "lieux": "lieux",
             "usagers": "usagers", "vehicules": "vehicules"}

dfs = {}
for name, keyword in file_keys.items():
    url = find_url(csv_resources, keyword, ANNEE)
    print(f"↓ {name:10s} : {url.split('/')[-1]}")
    dfs[name] = pd.read_csv(url, sep=";", encoding="latin1", low_memory=False)

caract, lieux, usagers, vehicules = dfs["caract"], dfs["lieux"], dfs["usagers"], dfs["vehicules"]

print("\nDimensions des fichiers :")
for k, v in dfs.items():
    print(f"  {k:10s} → {v.shape[0]:>7,} lignes × {v.shape[1]} colonnes")

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Interprétation — volumes</b><br><br>Les quatre fichiers ne comptent pas le même nombre de lignes car chacun a sa granularité propre : un accident peut impliquer plusieurs lieux (ex : carrefour), plusieurs véhicules, et plusieurs usagers. Le fichier <code>usagers</code> est donc le plus volumineux (≈126 000 victimes) et c'est à cette granularité qu'on se place pour modéliser la gravité individuelle.</div>

## 4. Nettoyage et enrichissement
<a id="4-nettoyage-et-enrichissement"></a>

Les étapes suivantes sont nécessaires :
- Conversion des coordonnées GPS (virgule française → point décimal).
- Extraction de l'heure depuis `hrmn`.
- Reconstruction d'une date complète pour obtenir le **jour de la semaine**.
- Calcul de l'âge du conducteur (année d'accident - année de naissance).
- Création de labels lisibles pour les variables catégorielles.
- Construction de la cible binaire `grave` (1 = tué ou blessé hospitalisé).

In [ ]:
# --- Coordonnées GPS ---
for col in ["lat", "long"]:
    caract[col] = pd.to_numeric(
        caract[col].astype(str).str.replace(",", ".").str.strip(),
        errors="coerce"
    )

# --- Heure ---
caract["heure"] = pd.to_numeric(
    caract["hrmn"].astype(str).str[:2], errors="coerce"
)

# --- Date + jour de la semaine ---
caract["date"] = pd.to_datetime(
    dict(year=caract["an"], month=caract["mois"], day=caract["jour"]),
    errors="coerce"
)
# lundi=0 ... dimanche=6
caract["jour_semaine"] = caract["date"].dt.dayofweek
caract["weekend"] = caract["jour_semaine"].isin([5, 6]).astype(int)

# --- Âge des usagers ---
usagers["age"] = int(ANNEE) - pd.to_numeric(usagers["an_nais"], errors="coerce")
# on écarte les âges aberrants
usagers.loc[(usagers["age"] < 0) | (usagers["age"] > 110), "age"] = np.nan

# --- Tranches d'âge ---
bins = [0, 14, 17, 24, 34, 44, 54, 64, 74, 110]
labels = ["0-14", "15-17", "18-24", "25-34", "35-44",
          "45-54", "55-64", "65-74", "75+"]
usagers["tranche_age"] = pd.cut(usagers["age"], bins=bins, labels=labels, right=True)

# --- Cible binaire ---
grav_labels = {1: "Indemne", 2: "Tué",
               3: "Blessé hospitalisé", 4: "Blessé léger"}
usagers["grav_label"] = usagers["grav"].map(grav_labels)
usagers["grave"] = usagers["grav"].isin([2, 3]).astype(int)

# --- Fusion principale (usager + accident + lieu) ---
df = (usagers
      .merge(caract, on="Num_Acc", how="left")
      .merge(lieux,  on="Num_Acc", how="left"))

print(f"Base analytique : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période couverte : {df['date'].min().date()} → {df['date'].max().date()}")
df[["Num_Acc", "grav_label", "grave", "age", "tranche_age",
    "heure", "jour_semaine"]].head()

## 5. Exploration statistique générale
<a id="5-exploration-statistique-générale"></a>

In [ ]:
# Vue synthétique
apercu = pd.DataFrame({
    "Métrique": [
        "Nombre d'accidents",
        "Nombre de véhicules impliqués",
        "Nombre d'usagers impliqués",
        "Nombre de tués",
        "Nombre de blessés hospitalisés",
        "Nombre de blessés légers",
        "Nombre d'indemnes",
        "Taux de mortalité (‰ victimes)",
        "Taux d'accidents graves (%)",
    ],
    "Valeur 2023": [
        f"{caract.shape[0]:,}",
        f"{vehicules.shape[0]:,}",
        f"{usagers.shape[0]:,}",
        f"{(usagers['grav']==2).sum():,}",
        f"{(usagers['grav']==3).sum():,}",
        f"{(usagers['grav']==4).sum():,}",
        f"{(usagers['grav']==1).sum():,}",
        f"{(usagers['grav']==2).mean()*1000:.1f}",
        f"{usagers['grave'].mean()*100:.1f}",
    ],
})
apercu

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Ce que disent ces chiffres</b><br><br>Sur environ 54 800 accidents corporels en 2023, on dénombre près de 126 000 usagers impliqués, dont <b>43 % sont indemnes</b>, <b>40 % blessés légèrement</b>, <b>15 % hospitalisés</b> et <b>2,4 % décédés</b>. La cible <code>grave</code> (hospitalisés + tués) concerne donc environ 17 % des victimes — c'est un problème de classification <b>déséquilibré</b>, point crucial pour la modélisation.</div>

In [ ]:
# Valeurs manquantes sur les variables clés
key_vars = ["lat", "long", "lum", "atm", "col", "catr", "circ",
            "surf", "situ", "age", "sexe", "secu1"]
miss = (df[key_vars].isna().mean()*100).round(2).sort_values(ascending=False)
miss = miss.reset_index()
miss.columns = ["Variable", "% manquants"]
miss

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Qualité des données</b><br><br>La plupart des variables présentent un taux de valeurs manquantes faible. Les coordonnées GPS (<code>lat</code>, <code>long</code>) et certaines variables de conditions de circulation peuvent être ponctuellement absentes — nous les écarterons via <code>dropna()</code> au moment de la modélisation pour éviter tout biais d'imputation.</div>

## 6. Analyses univariées
<a id="6-analyses-univariées"></a>

### 6.1 Temporelles (mois, jour, heure)
<a id="61-temporelles-mois-jour-heure"></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# --- Mois ---
monthly = caract["mois"].value_counts().sort_index()
axes[0].bar(monthly.index, monthly.values, color=sns.color_palette("Blues_d", 12))
axes[0].set_title("Nombre d'accidents par mois")
axes[0].set_xlabel("Mois"); axes[0].set_ylabel("Accidents")
axes[0].set_xticks(range(1, 13))

# --- Jour de la semaine ---
jour_labels = ["Lun","Mar","Mer","Jeu","Ven","Sam","Dim"]
weekly = caract["jour_semaine"].value_counts().sort_index()
colors_w = ["#4a90e2"]*5 + ["#e74c3c"]*2
axes[1].bar(weekly.index, weekly.values, color=colors_w)
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(jour_labels)
axes[1].set_title("Accidents par jour de la semaine")
axes[1].set_xlabel(""); axes[1].set_ylabel("Accidents")

# --- Heure ---
hourly = caract["heure"].value_counts().sort_index()
h_idx = hourly.index.to_numpy()
h_val = hourly.values
axes[2].plot(h_idx, h_val, color="#2c3e50", linewidth=2, marker="o", markersize=4)
axes[2].fill_between(h_idx, h_val, alpha=0.25, color="#3498db")
axes[2].set_title("Accidents par heure de la journée")
axes[2].set_xlabel("Heure"); axes[2].set_ylabel("Accidents")
axes[2].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Interprétation — dimensions temporelles</b><br><br><b>Saisonnalité</b> : on observe un creux net en février (jours courts, moins de déplacements) puis un pic en juin et octobre, mois de forte mobilité (fin d'année scolaire, rentrée). <br><br><b>Jour de la semaine</b> : les accidents sont répartis de façon relativement homogène en semaine ; le <b>vendredi</b> est légèrement supérieur (début de week-end, déplacements loisirs). Samedi et dimanche sont plus bas <i>en volume</i>, mais on verra plus loin qu'ils sont plus graves. <br><br><b>Heure</b> : courbe bimodale classique — pic du matin (8 h, trajet domicile-travail) et pic marqué du soir (17 h–18 h). La nuit est nettement moins accidentogène en volume, mais on sait qu'elle concentre la gravité.</div>

In [ ]:
# Zoom sur la gravité par heure
grav_heure = df.groupby("heure")["grave"].mean() * 100

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(grav_heure.index, grav_heure.values,
        color="#c0392b", linewidth=2.2, marker="o", markersize=5)
ax.fill_between(grav_heure.index, grav_heure.values, alpha=0.2, color="#e74c3c")
ax.axhline(df["grave"].mean()*100, color="gray", linestyle="--",
           label=f"Moyenne nationale ({df['grave'].mean()*100:.1f} %)")
ax.set_title("Part des accidents graves (tués + hospitalisés) par heure")
ax.set_xlabel("Heure"); ax.set_ylabel("% d'accidents graves")
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout(); plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 La nuit amplifie la gravité</b><br><br>La <b>part d'accidents graves</b> grimpe nettement entre 22 h et 6 h du matin. Moins d'accidents en volume, mais une proportion de tués/hospitalisés sensiblement plus élevée — un effet combiné de la fatigue, de l'alcool, d'une moindre visibilité et de vitesses moyennes plus élevées en circulation nocturne.</div>

### 6.2 Conditions environnementales
<a id="62-conditions-environnementales"></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Météo
atm_labels = {1:"Normale",2:"Pluie légère",3:"Pluie forte",4:"Neige/grêle",
              5:"Brouillard",6:"Vent fort",7:"Éblouissement",8:"Temps couvert",9:"Autre"}
atm = caract["atm"].map(atm_labels).value_counts()
sns.barplot(x=atm.values, y=atm.index, ax=axes[0, 0], palette="coolwarm")
axes[0, 0].set_title("Conditions météo")
axes[0, 0].set_xlabel("Nombre d'accidents")

# Luminosité
lum_labels = {1:"Plein jour",2:"Crépuscule",3:"Nuit sans éclairage",
              4:"Nuit éclairage allumé",5:"Nuit éclairage éteint"}
lum = caract["lum"].map(lum_labels).value_counts()
sns.barplot(x=lum.values, y=lum.index, ax=axes[0, 1], palette="magma_r")
axes[0, 1].set_title("Conditions de luminosité")
axes[0, 1].set_xlabel("Nombre d'accidents")

# Surface
surf_labels = {1:"Normale",2:"Mouillée",3:"Flaques",4:"Inondée",
               5:"Enneigée",6:"Boue",7:"Verglacée",8:"Corps gras/huile",9:"Autre"}
surf = lieux["surf"].map(surf_labels).value_counts()
sns.barplot(x=surf.values, y=surf.index, ax=axes[1, 0], palette="Blues_r")
axes[1, 0].set_title("État de la surface")
axes[1, 0].set_xlabel("Nombre d'accidents")

# Type de collision
col_labels = {1:"Frontale (2 véh.)",2:"Arrière (2 véh.)",3:"Côté (2 véh.)",
              4:"Chaîne (3+)",5:"Multiple",6:"Autre",7:"Sans collision"}
col = caract["col"].map(col_labels).value_counts()
sns.barplot(x=col.values, y=col.index, ax=axes[1, 1], palette="YlOrRd")
axes[1, 1].set_title("Type de collision")
axes[1, 1].set_xlabel("Nombre d'accidents")

plt.tight_layout(); plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Environnement — conditions habituelles, accidents habituels</b><br><br>La grande majorité des accidents se produisent par <b>conditions normales</b> (météo clémente, chaussée sèche, plein jour). C'est un effet d'exposition : la circulation est dense dans ces conditions. Les situations exceptionnelles (neige, verglac, brouillard) représentent un volume faible, mais leur gravité relative est souvent plus élevée. <br><br>Côté collisions, <b>les chocs latéraux et par l'arrière</b> dominent — typiques des zones urbaines (carrefours, ralentissements).</div>

### 6.3 Caractéristiques des usagers
<a id="63-caractéristiques-des-usagers"></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# Sexe
sexe_labels = {1:"Masculin", 2:"Féminin"}
sexe = usagers["sexe"].map(sexe_labels).value_counts()
axes[0].bar(sexe.index, sexe.values, color=["#3498db", "#e91e8c"])
axes[0].set_title("Victimes par sexe")
for i, v in enumerate(sexe.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

# Catégorie d'usager
catu_labels = {1:"Conducteur", 2:"Passager", 3:"Piéton"}
catu = usagers["catu"].map(catu_labels).value_counts()
axes[1].bar(catu.index, catu.values, color=["#2ecc71", "#f39c12", "#9b59b6"])
axes[1].set_title("Catégorie d'usager")
for i, v in enumerate(catu.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

# Distribution des âges
axes[2].hist(usagers["age"].dropna(), bins=40, color="#34495e",
             edgecolor="white", alpha=0.85)
axes[2].axvline(usagers["age"].median(), color="red", linestyle="--",
                label=f"Médiane = {usagers['age'].median():.0f} ans")
axes[2].set_title("Distribution des âges des victimes")
axes[2].set_xlabel("Âge"); axes[2].set_ylabel("Effectif")
axes[2].legend()

plt.tight_layout(); plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Portrait des usagers accidentés</b><br><br>Les victimes sont <b>majoritairement des hommes</b> (~65 %) — un écart structurel observé dans tous les pays : les hommes conduisent plus, plus longtemps et adoptent plus souvent des comportements à risque (vitesse, alcool).<br><br>Les <b>conducteurs</b> sont la première catégorie de victimes, suivis des passagers et des piétons. La distribution des âges montre une <b>forte concentration entre 18 et 34 ans</b> : c'est l'âge du permis récent et de l'exposition maximale. La médiane se situe autour de 35 ans.</div>

In [ ]:
# Tranches d'âge par catégorie d'usager
tab = pd.crosstab(usagers["tranche_age"], usagers["catu"].map(catu_labels))
tab = tab[["Conducteur", "Passager", "Piéton"]]

ax = tab.plot(kind="bar", stacked=True, figsize=(11, 4.5),
              color=["#2ecc71", "#f39c12", "#9b59b6"], edgecolor="white")
ax.set_title("Répartition des victimes par tranche d'âge et catégorie d'usager")
ax.set_xlabel("Tranche d'âge"); ax.set_ylabel("Nombre de victimes")
ax.legend(title="Catégorie")
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
tab

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Qui est où ?</b><br><br>Les <b>jeunes (18-24 ans)</b> sont très majoritairement victimes en tant que <b>conducteurs</b> — premières années après le permis. Les <b>enfants (0-14 ans)</b> sont surtout victimes comme passagers ou piétons. La part des piétons augmente chez les <b>plus de 65 ans</b> — population plus vulnérable et moins protégée par un habitacle.</div>

### 6.4 Équipements de sécurité
<a id="64-équipements-de-sécurité"></a>

In [ ]:
secu_labels = {
    1:"Ceinture", 2:"Casque", 3:"Dispositif enfant",
    4:"Gilet réfléchissant", 5:"Airbag (2RM)",
    6:"Gants (2RM)", 7:"Gants+airbag", 8:"Non déterminable", 9:"Autre"
}
secu = usagers["secu1"].map(secu_labels).value_counts().head(8)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(x=secu.values, y=secu.index, palette="crest", ax=ax)
ax.set_title("Équipement de sécurité principal des victimes")
ax.set_xlabel("Nombre de victimes")
plt.tight_layout(); plt.show()

# Gravité selon le port de la ceinture (sous-population en voiture)
df_voit = df[df["catu"].isin([1, 2])]  # conducteurs + passagers
ceinture = df_voit.groupby(df_voit["secu1"]==1)["grave"].mean() * 100
ceinture.index = ["Sans ceinture", "Avec ceinture"]
print("Part d'accidents graves (en voiture) :")
print(ceinture.round(1).to_string())

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 La ceinture, une évidence statistique</b><br><br>La ceinture est de très loin l'équipement le plus recensé. Sur les usagers de voiture, la <b>part d'accidents graves est significativement plus élevée chez ceux sans ceinture</b>, résultat cohérent avec toute la littérature en sécurité routière. Cette variable sera naturellement retenue dans le modèle.</div>

## 7. Analyses bivariées (croisements avec la gravité)
<a id="7-analyses-bivariées-croisements-avec-la-gravité"></a>

On regarde maintenant la part des accidents graves selon différentes dimensions.
L'objectif est d'identifier les variables les plus <b>discriminantes</b>, utiles pour
la modélisation.

In [ ]:
def bar_gravite(var, mapping, title, ax, palette="viridis"):
    """Bar chart du % d'accidents graves par modalité."""
    tmp = df.copy()
    tmp[var+"_lab"] = tmp[var].map(mapping)
    pct = tmp.groupby(var+"_lab")["grave"].mean().sort_values() * 100
    sns.barplot(x=pct.values, y=pct.index, ax=ax, palette=palette)
    ax.axvline(df["grave"].mean()*100, color="red", linestyle="--",
               label=f"Moyenne ({df['grave'].mean()*100:.1f} %)")
    ax.set_title(title); ax.set_xlabel("% d'accidents graves")
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

catr_labels = {1:"Autoroute",2:"Route nationale",3:"Départementale",
               4:"Voie communale",5:"Hors réseau public",6:"Parc stationnement",
               7:"Voie privée",9:"Autre"}
bar_gravite("catr", catr_labels, "Gravité par type de route", axes[0, 0], "Blues_d")
bar_gravite("atm", {1:"Normale",2:"Pluie légère",3:"Pluie forte",4:"Neige",
                    5:"Brouillard",7:"Éblouissement",8:"Couvert"},
            "Gravité par météo", axes[0, 1], "coolwarm")
bar_gravite("lum", {1:"Plein jour",2:"Crépuscule",3:"Nuit s/ éclair.",
                    4:"Nuit éclair.",5:"Nuit ét."},
            "Gravité par luminosité", axes[1, 0], "magma_r")
bar_gravite("catu", {1:"Conducteur",2:"Passager",3:"Piéton"},
            "Gravité par catégorie d'usager", axes[1, 1], "Set2")

plt.tight_layout(); plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Les variables les plus discriminantes</b><br><br><b>Type de route</b> : les <b>routes nationales et départementales</b> sont 2 à 3 fois plus graves que les voies urbaines — vitesses supérieures, absence d'obstacles amortisseurs. <br><br><b>Météo</b> : le <b>brouillard</b> et la <b>neige</b> ressortent au-dessus de la moyenne. Effet visibilité et distance de freinage. <br><br><b>Luminosité</b> : la <b>nuit sans éclairage</b> est de loin la configuration la plus grave. <br><br><b>Catégorie d'usager</b> : les <b>piétons</b> ont une part d'accidents graves nettement plus élevée que les automobilistes — pas de carrosserie pour amortir le choc. Ces quatre variables seront toutes intégrées au modèle.</div>

In [ ]:
# Gravité par tranche d'âge et sexe
pct_age_sexe = (df.groupby(["tranche_age", "sexe"])["grave"].mean() * 100).unstack()
pct_age_sexe.columns = pct_age_sexe.columns.map({1: "Hommes", 2: "Femmes"})

fig, ax = plt.subplots(figsize=(11, 4.5))
pct_age_sexe.plot(kind="bar", ax=ax, color=["#3498db", "#e91e8c"], edgecolor="white")
ax.axhline(df["grave"].mean()*100, color="gray", linestyle="--",
           label=f"Moyenne ({df['grave'].mean()*100:.1f} %)")
ax.set_title("% d'accidents graves par tranche d'âge et sexe")
ax.set_xlabel("Tranche d'âge"); ax.set_ylabel("% graves")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
pct_age_sexe.round(1)

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 L'âge, un facteur aggravant aux extrêmes</b><br><br>La gravité dessine une <b>courbe en U</b> : les <b>enfants de moins de 14 ans</b> et surtout les <b>seniors (65+)</b> ont des accidents plus graves. Les jeunes adultes (18-34 ans) ont beaucoup d'accidents mais moins souvent graves — leur organisme résiste mieux aux chocs. À partir de 65 ans, la <b>fragilité physiologique</b> transforme un choc modéré en blessure hospitalisable. L'écart homme/femme existe mais reste modeste par rapport à l'effet âge.</div>

## 8. Cartographie
<a id="8-cartographie"></a>

### 8.1 Heatmap nationale
<a id="81-heatmap-nationale"></a>

Une heatmap permet d'identifier les zones de forte densité d'accidents. On se limite
à la France métropolitaine pour la lisibilité.

In [ ]:
coords = caract[["lat", "long"]].dropna()
coords = coords[(coords["lat"].between(41, 52)) & (coords["long"].between(-5, 10))]
sample = coords.sample(min(5000, len(coords)), random_state=42)

m_heat = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB positron")
HeatMap(sample[["lat","long"]].values.tolist(),
        radius=8, blur=12, min_opacity=0.4).add_to(m_heat)
m_heat.save("outputs/carte_heatmap.html")
m_heat

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Sans surprise, la géographie des accidents épouse celle de la population</b><br><br>Les foyers les plus intenses se concentrent en <b>Île-de-France</b>, sur la façade <b>méditerranéenne</b> (Marseille, Nice), à <b>Lyon</b>, <b>Bordeaux</b>, <b>Toulouse</b> et le long des grands axes autoroutiers. La densité d'accidents suit avant tout la densité de circulation — ce qui motive l'analyse département par département dans la section suivante.</div>

### 8.2 Choroplèthe par département
<a id="82-choroplèthe-par-département"></a>

Pour neutraliser l'effet population, on visualise le nombre d'accidents par département.
On récupère les contours départementaux via le portail **France-GeoJSON**.

In [ ]:
# Nombre d'accidents par département
dep_counts = caract["dep"].astype(str).str.zfill(2).value_counts().to_dict()

# Récupération des contours départementaux
geojson_url = ("https://raw.githubusercontent.com/gregoiredavid/"
               "france-geojson/master/departements.geojson")
dep_df = pd.DataFrame(
    [(k, v) for k, v in dep_counts.items()],
    columns=["dep", "nb_accidents"]
)

m_chor = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=geojson_url,
    data=dep_df,
    columns=["dep", "nb_accidents"],
    key_on="feature.properties.code",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Nombre d'accidents 2023",
    nan_fill_color="white",
).add_to(m_chor)
m_chor.save("outputs/carte_choroplethe.html")
m_chor

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Top 10 des départements les plus accidentogènes</b><br><br><b>Paris (75)</b> est en tête, suivi des départements de la petite couronne (93, 92, 94). Viennent ensuite les <b>Bouches-du-Rhône (13)</b>, le <b>Rhône (69)</b>, la <b>Gironde (33)</b> et le <b>Nord (59)</b>. L'urbanisation est la première variable explicative en volume — mais on notera qu'un département rural peu peuplé peut afficher une <b>mortalité relative</b> bien supérieure (accidents plus graves sur les routes à grande vitesse). Une analyse normalisée par population complèterait utilement.</div>

### 8.3 Accidents mortels (cluster)
<a id="83-accidents-mortels-cluster"></a>

Focus sur les accidents ayant fait au moins un décès, avec regroupement dynamique.

In [ ]:
# Accidents ayant fait au moins un tué
acc_mortels_ids = usagers.loc[usagers["grav"]==2, "Num_Acc"].unique()
acc_mortels = caract[caract["Num_Acc"].isin(acc_mortels_ids)].dropna(subset=["lat","long"])
acc_mortels = acc_mortels[(acc_mortels["lat"].between(41, 52)) &
                          (acc_mortels["long"].between(-5, 10))]
print(f"{len(acc_mortels):,} accidents mortels géolocalisés")

m_mort = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB dark_matter")
cluster = MarkerCluster().add_to(m_mort)

for _, row in acc_mortels.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["long"]],
        radius=3, color="#e74c3c", fill=True, fill_opacity=0.7,
        popup=f"Dép. {row['dep']} — {row['date'].date() if pd.notna(row['date']) else ''}"
    ).add_to(cluster)

m_mort.save("outputs/carte_mortels.html")
m_mort

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Les accidents mortels sont, eux, plus ruraux</b><br><br>Contrairement à la heatmap générale, la carte des <b>accidents mortels</b> met en avant les grands axes de province et les zones rurales. En ville, les chocs à basse vitesse sont rarement mortels ; sur route nationale ou départementale à 80/90 km/h, ils le sont beaucoup plus souvent. La <b>France intérieure</b> apparaît plus clairement qu'en heatmap brute.</div>

## 9. Modélisation — Prédire la gravité d'un accident
<a id="9-modélisation--prédire-la-gravité-dun-accident"></a>

### 9.1 Préparation
<a id="91-préparation"></a>

On cherche à prédire **`grave` ∈ {0, 1}** à partir de variables connues au moment de
l'accident (conditions, route, usager). La cible est déséquilibrée (17 % de 1) — on
utilisera `class_weight="balanced"` et on surveillera le <b>recall</b> sur la classe
positive, plus critique que la précision pour un enjeu de santé publique.

In [ ]:
features = ["lum", "atm", "col", "heure", "mois", "jour_semaine",
            "catr", "circ", "surf", "situ", "catu", "sexe", "age", "secu1"]
target = "grave"

ml = df[features + [target]].dropna().copy()
print(f"Dataset ML : {ml.shape[0]:,} lignes")
print(f"Répartition : {ml[target].value_counts().to_dict()}")
print(f"Taux de graves : {ml[target].mean()*100:.1f} %")

X = ml[features]
y = ml[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

### 9.2 Régression logistique (baseline)
<a id="92-régression-logistique-baseline"></a>

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

logreg = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=42
)
logreg.fit(X_train_s, y_train)
y_pred_lr   = logreg.predict(X_test_s)
y_proba_lr  = logreg.predict_proba(X_test_s)[:, 1]

print("=== Régression logistique (baseline) ===")
print(classification_report(y_test, y_pred_lr,
                            target_names=["Léger/Indemne", "Grave"]))
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_lr):.3f}")

### 9.3 Random Forest équilibrée
<a id="93-random-forest-équilibrée"></a>

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest équilibrée ===")
print(classification_report(y_test, y_pred_rf,
                            target_names=["Léger/Indemne", "Grave"]))
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_rf):.3f}")

### 9.4 Comparaison des modèles
<a id="94-comparaison-des-modèles"></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Matrice de confusion — Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=["Léger", "Grave"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion — Régression logistique")

# Matrice de confusion — Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=["Léger", "Grave"]).plot(
    ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title("Confusion — Random Forest")

# Courbes ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
axes[2].plot(fpr_lr, tpr_lr, label=f"LogReg  AUC={roc_auc_score(y_test,y_proba_lr):.3f}",
             color="#3498db", linewidth=2)
axes[2].plot(fpr_rf, tpr_rf, label=f"RForest AUC={roc_auc_score(y_test,y_proba_rf):.3f}",
             color="#27ae60", linewidth=2)
axes[2].plot([0, 1], [0, 1], "--", color="gray")
axes[2].set_xlabel("Taux de faux positifs"); axes[2].set_ylabel("Taux de vrais positifs")
axes[2].set_title("Courbes ROC")
axes[2].legend()

plt.tight_layout(); plt.show()

# Tableau récap
recap = pd.DataFrame({
    "Modèle":   ["Régression logistique", "Random Forest"],
    "Accuracy": [(y_pred_lr==y_test).mean(), (y_pred_rf==y_test).mean()],
    "F1 (grave)": [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf)],
    "ROC-AUC":  [roc_auc_score(y_test, y_proba_lr),
                 roc_auc_score(y_test, y_proba_rf)],
}).round(3)
recap

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Interprétation des modèles</b><br><br>Les deux modèles obtiennent une <b>AUC comparable (~0.70)</b>, la forêt aléatoire étant légèrement supérieure grâce à sa capacité à capter des interactions non linéaires. Point important : en équilibrant les classes (<code>class_weight='balanced'</code>), on obtient un <b>recall sur la classe grave bien supérieur</b> au modèle initial non équilibré (où le recall n'était que de 9 %), au prix d'une baisse de la précision globale. <br><br>Dans un contexte de santé publique, <b>détecter davantage d'accidents potentiellement graves est préférable</b> à maximiser l'accuracy — un faux positif coûte peu ; un faux négatif, beaucoup.</div>

### 9.5 Importance des variables
<a id="95-importance-des-variables"></a>

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features) \
                .sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = sns.color_palette("viridis", len(importances))
ax.barh(importances.index, importances.values, color=colors)
ax.set_title("Importance des variables — Random Forest")
ax.set_xlabel("Score d'importance")
for i, v in enumerate(importances.values):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

<div style="background-color:#e7f3fe; border-left:6px solid #2196F3; padding:14px 18px; margin:12px 0; border-radius:4px; font-family:Segoe UI, Arial, sans-serif; color:#0b3954;"><b>💡 Quelles variables pèsent le plus ?</b><br><br>Les variables dominantes sont <b>l'âge de l'usager</b>, <b>l'heure</b>, <b>l'équipement de sécurité (secu1)</b>, <b>le type de route (catr)</b> et <b>la catégorie d'usager (catu)</b>. Ce résultat est cohérent avec les analyses descriptives précédentes : la fragilité liée à l'âge, l'effet nuit, le port de la ceinture, le type de voirie (urbain vs. vitesse élevée) et la vulnérabilité des piétons ressortent de façon robuste. <br><br>À l'inverse, le mois et le jour de la semaine ont un pouvoir prédictif marginal — la saisonnalité influence le volume d'accidents mais peu leur gravité conditionnelle.</div>

## 10. Conclusion
<a id="10-conclusion"></a>

<div style="background-color:#e8f5e9; border-left:6px solid #4caf50; padding:14px 18px; margin:12px 0; border-radius:4px; color:#1b5e20;"><b>📌 Synthèse</b><br><br>Cette analyse des accidents corporels de la route en France en 2023 a permis de :<br><br>• <b>Caractériser</b> la sinistralité sur les dimensions temporelle, spatiale, environnementale et humaine.<br>• <b>Identifier</b> les facteurs les plus associés à la gravité : nuit, âge élevé, routes nationales/départementales, piétons, absence de ceinture.<br>• <b>Cartographier</b> trois lectures complémentaires — densité brute, intensité départementale, localisation des accidents mortels.<br>• <b>Modéliser</b> la gravité avec une régression logistique et une forêt aléatoire, en traitant explicitement le déséquilibre des classes.<br><br><b>Limites.</b> Le modèle atteint une AUC d'environ 0.70 — utile mais non suffisant pour un usage opérationnel. Des variables <b>comportementales</b> (alcool, vitesse au moment du choc, usage du téléphone) ne sont pas dans la base BAAC et bénéficieraient grandement au pouvoir prédictif. Par ailleurs, cette analyse est descriptive : les associations mises en évidence ne sont pas des relations causales.<br><br><b>Pistes d'extension.</b> Intégrer les données météo fines d'Infoclimat/Météo-France, croiser avec le trafic quotidien des axes routiers (données SIREDO), construire un modèle XGBoost avec optimisation d'hyperparamètres, ou développer un dashboard Streamlit interactif.</div>

---
### 📚 Sources
- **Données BAAC 2023** : [data.gouv.fr](https://www.data.gouv.fr/fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2023/)
- **Fonds de carte GeoJSON** : [france-geojson](https://github.com/gregoiredavid/france-geojson)
- **ONISR** — Bilan annuel de la sécurité routière
- Cours *Python pour la Data Science* — Lino Galiana (ENSAE), 2025-2026

---
<p style="text-align:center; color:#777; font-size:0.9em;">
Notebook réalisé dans le cadre du projet de fin d'année —
Sécurité routière en France 2023.
</p>